# Codes for the pipeline of obtaining plots from saved model checkpoints


### First, we will be getting the covariance matrices from the saved model checkpoints

Run these helper functions and library importing codes before anything else.

In [ ]:
import os
import PIL
import torch
import torchvision
import tqdm
import BayesCompare as bc
import pickle
import numpy as np
import json
import matplotlib.pyplot as plt
from matplotlib import gridspec
from pathlib import Path
import seaborn as sns
import pandas as pd
import re

home_path = Path.home()

def get_specific_layers(layers_names_list, wanted_layers):

    output_layer_names = []

    for i, layer in enumerate(layers_names_list):

        if any(w in layer for w in wanted_layers):

            if "linear" in layer:

                if i <= (len(layers_names_list) - 4):

                    if "transpose" in layers_names_list[i + 2]:

                        output_layer_names.append(layers_names_list[i + 2])

                    else:

                        output_layer_names.append(layer)

                else:

                    output_layer_names.append(layer)

            else:

                output_layer_names.append(layer)

    return output_layer_names


def load_model(model_name, **kwargs):

    dir = kwargs.get("dir", None)

    if dir:
        snapshot = torch.load(dir)
        model = torchvision.models.get_model(model_name, weights=None)
        model.load_state_dict(snapshot["model"])

    else:
        model = torchvision.models.get_model(model_name, weights=kwargs["weights"])

    return model.eval()

pattern = re.compile(r"^([a-zA-Z]+)_(\d{1,3})_(\d{1,3})$")

def split_string(s):
    m = pattern.match(s)
    if not m:
        return [None, None, None]
    return m.group(1), m.group(2), m.group(3)

Load the image dataset you will be using for obtaining the covariances. For me this is MS COCO. Then, apply images the transforms that your models require.

In [ ]:
im_folder = os.path.join(
    home_path, "Documents/BayesCompare/images/unlabeled2017"
)  # Put your path to COCO images here
file_names = os.listdir(im_folder)

N = 1000
ims = [PIL.Image.open(os.path.join(im_folder, f_name)) for f_name in file_names[:N]]

transforms = torchvision.models.ViT_B_16_Weights.IMAGENET1K_V1.transforms()

transformed_ims = [transforms(im.convert("RGB")) for im in ims]
x_input = torch.stack(transformed_ims)

In [ ]:
model_kwargs = [
    {
        "dir": os.path.join(home_path, "Documents/BayesCompare/checkpoints/ViT_b16/snapshot_ep199_seed111_iter4.pth")
    },
    {
        "dir": os.path.join(home_path, "Documents/BayesCompare/checkpoints/ViT_b16/snapshot_ep199_seed222_iter6.pth")
    },
    {
        "dir": os.path.join(home_path, "Documents/BayesCompare/checkpoints/ViT_b16/snapshot_ep200_seed888_iter4.pth")
    },
    {
        "dir": os.path.join(home_path, "Documents/BayesCompare/checkpoints/ViT_b16/snapshot_ep200_seed999_iter4.pth")
    },
    {"weights": torchvision.models.ViT_B_16_Weights.IMAGENET1K_V1},
]

# Specify layers to get the covs for
layer_types = ["linear", "add", "gelu", "attention", "layernorm", "conv"]

models_covs_list = []

# For each model, get the covs_dict for all wanted layers and collect them into the models_covs_list
for model_kwarg in tqdm.tqdm(model_kwargs, desc="Models", position=1):

    model = load_model("vit_b_16", **model_kwarg)

    all_layer_names = bc.get_layer_names(model)

    wanted_layers = get_specific_layers(all_layer_names, layer_types)

    cov_dict = bc.cov_extractor(model, wanted_layers, x_input)

    models_covs_list.append(cov_dict)

# Save the models_covs_list which have all the wanted activations for all models
with open("covs_1000_vitb16_inferencemode.pkl", "wb") as f:
    pickle.dump(models_covs_list, f)

Now, we have the covariances. But we need to apply trace normalization and noise addition to them before computing the distances. This is because our distance comuting function `dist_parallel` expects the covariance matrices to be trace normalized.

In [ ]:
# Load the saved covs
with open(
    os.path.join(home_path, "Documents/BayesCompare/covs_1000_vitb16_inferencemode.pkl"), "rb"
) as f:
    covs_dicts = pickle.load(f)

covs = []

for cov_dict in covs_dicts:

    covs.append(list(cov_dict.values()))
    layer_names = list(cov_dict.keys())

covs = np.stack(covs)
covs = covs.reshape(covs.shape[0] * covs.shape[1], covs.shape[2], covs.shape[3])

# Normalize them
normalized_covs = []

for cov in tqdm.tqdm(covs):
    normalized_covs.append(
        bc.cov_utils.cov_sigma(bc.cov_utils.trace_norm(cov), noise_var=10 / 11)
    )

# Save the normalized covs
np.save(
    os.path.join(home_path, "Documents/BayesCompare/covs_1000_vitb16_inferencemode_normalized.npy"),
    normalized_covs,
)

### Second, we will be computing the distances from the trace normalized covariances.

In [ ]:
bc.measure_dist_parallel(
    covs_dir=os.path.join(home_path, "Documents/BayesCompare/covs_1000_vitb16_inferencemode_normalized.npy"),
    output_dir=os.path.join(home_path, "Documents/BayesCompare/outputs/"),
    meas_name=["wasserstein", "JSD"],
)

### Next step is to obtain different plots.

We also need to apply a small preprocessing to the distance matrices. Their diagonals are `NaN`, we need to convert them to `0.0`s to obtain sensible plots. This step may become unnecessary as this should be fixed inside the distance calculating function.

In [ ]:
import h5py

wass_dist_dir = os.path.join(home_path, "Documents/BayesCompare/outputs/dist_covs_1000_vitb16_inferencemode_normalized_wasserstein.hdf5")
jsd_dist_dir = os.path.join(home_path, "Documents/BayesCompare/outputs/dist_covs_1000_vitb16_inferencemode_normalized_JSD.hdf5")

with h5py.File(wass_dist_dir, "r") as f:

    wass_dist = f["dist"][...]

with h5py.File(jsd_dist_dir, "r") as f:

    jsd_dist = f["dist"][...]

# Replace the nan values in the diagonal of the distance matrices with 0.0
for i in range(len(wass_dist)):
    wass_dist[i, i] = 0.0
    jsd_dist[i, i] = 0.0

# Then, save them as npy files
np.save(
    os.path.join(home_path, "Documents/BayesCompare/outputs/dist_covs_1000_vitb16_inferencemode_normalized_wasserstein.npy"),
    wass_dist,
)
np.save(
    os.path.join(home_path, "Documents/BayesCompare/outputs/dist_covs_1000_vitb16_inferencemode_normalized_JSD.npy"),
    jsd_dist,
)

We can obtain now our different plots from the distance matrices that are saved as numpy arrays.

##### 1.1- All Models, All Layers Matrix Plot

In [ ]:
# Load them if necessary
# wass_dist = np.load(os.path.join(home_path, "Documents/BayesCompare/outputs/dist_covs_1000_vitb16_inferencemode_normalized_wasserstein.npy"))
# jsd_dist = np.load(os.path.join(home_path, "Documents/BayesCompare/outputs/dist_covs_1000_vitb16_inferencemode_normalized_JSD.npy"))

dists = [wass_dist, jsd_dist]
dist_names = ["wasserstein", "JSD"]

for i, dist_name in enumerate(dist_names):

    plt.figure(figsize=(12, 8), dpi=800)
    plt.imshow(dists[i], "bone", vmin=0, vmax=np.max(dists[i]))
    plt.colorbar()
    plt.title(dist_name.capitalize() + " Distance")
    ax = plt.gca()
    ax.set_axis_off()
    plt.savefig(
        os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/dist_")
        + dist_name
        + "_allvits_alllayers_inferencemode.svg",
        dpi=800,
    )

##### 1.2- Block Separated All Models, All Layers Matrix Plot

In [ ]:
# Load them if necessary
# wass_dist = np.load(os.path.join(home_path, "Documents/BayesCompare/outputs/dist_covs_1000_vitb16_inferencemode_normalized_wasserstein.npy"))
# jsd_dist = np.load(os.path.join(home_path, "Documents/BayesCompare/outputs/dist_covs_1000_vitb16_inferencemode_normalized_JSD.npy"))

dists = [wass_dist, jsd_dist]
dist_names = ["wasserstein", "JSD"]

boundaries = [99, 199, 299, 399]

N = dists[0].shape[0]
edges = [0] + boundaries + [N]
groups = [(edges[i], edges[i + 1]) for i in range(len(edges) - 1)]
num_groups = len(groups)
gap = 0.05  # spacing between blocks (tune this)
gs = gridspec.GridSpec(num_groups, num_groups, wspace=gap, hspace=gap)

for i, dist_name in enumerate(dist_names):

    plotting_dist = dists[i]
    
    fig = plt.figure(figsize=(12, 12), dpi=800)

    for i, (r0, r1) in enumerate(groups):
        for j, (c0, c1) in enumerate(groups):
            ax = fig.add_subplot(gs[i, j])
            block = plotting_dist[r0:r1, c0:c1]
            ax.imshow(block, cmap="bone", vmin=0, vmax=np.max(plotting_dist))
            ax.set_xticks([])
            ax.set_yticks([])

            # remove the box around the matrix plot
            for spine in ax.spines.values():
                spine.set_visible(False)
    
    ax = plt.gca()
    ax.set_axis_off()
    plt.savefig(
        os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/dist_")
        + dist_name
        + "_allvits_alllayers_inferencemode_blockseparated.svg",
        dpi=800,
    )

##### 1.3- Layer Specific All Models Matrix Plot

In [ ]:
# Load them if necessary
# wass_dist = np.load(
#     os.path.join(home_path, "Documents/BayesCompare/outputs/dist_covs_1000_vitb16_inferencemode_normalized_wasserstein_corrected.npy")
# )
# jsd_dist = np.load(
#     os.path.join(home_path, "Documents/BayesCompare/outputs/dist_covs_1000_vitb16_inferencemode_normalized_JSD_corrected.npy")
# )

dists = [wass_dist, jsd_dist]
dist_names = ["wasserstein", "JSD"]

# set this value specific for Wasserstein and JSD based on their overall colorbar scales from full Kornblith plots
colorbar_max_vals = [7.5, 1]

with open(
    os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/vit_inferencemode_layernames.json"),
    "r",
    encoding="utf-8",
) as f:
    model_layers = json.load(f)

for d, dist_name in enumerate(dist_names):

    m = 5  # number of models
    n = int(len(dists[d]) / m)  # number of layers per model

    layernorm_idx = []
    linear_idx = []
    add_idx = []
    gelu_idx = []
    attention_idx = []

    for i, layer_name in enumerate(model_layers):
        if "layernorm" in layer_name:
            layernorm_idx.append(i)
        elif "linear" in layer_name:
            linear_idx.append(i)
        elif "add" in layer_name:
            add_idx.append(i)
        elif "gelu" in layer_name:
            gelu_idx.append(i)
        elif "attention" in layer_name:
            attention_idx.append(i)

    layernorm_all = np.concatenate(
        [
            np.array(layernorm_idx),
            np.array(layernorm_idx) + 100,
            np.array(layernorm_idx) + 200,
            np.array(layernorm_idx) + 300,
            np.array(layernorm_idx) + 400,
        ]
    ).tolist()
    linear_all = np.concatenate(
        [
            np.array(linear_idx),
            np.array(linear_idx) + 100,
            np.array(linear_idx) + 200,
            np.array(linear_idx) + 300,
            np.array(linear_idx) + 400,
        ]
    ).tolist()
    add_all = np.concatenate(
        [
            np.array(add_idx),
            np.array(add_idx) + 100,
            np.array(add_idx) + 200,
            np.array(add_idx) + 300,
            np.array(add_idx) + 400,
        ]
    ).tolist()
    gelu_all = np.concatenate(
        [
            np.array(gelu_idx),
            np.array(gelu_idx) + 100,
            np.array(gelu_idx) + 200,
            np.array(gelu_idx) + 300,
            np.array(gelu_idx) + 400,
        ]
    ).tolist()
    attention_all = np.concatenate(
        [
            np.array(attention_idx),
            np.array(attention_idx) + 100,
            np.array(attention_idx) + 200,
            np.array(attention_idx) + 300,
            np.array(attention_idx) + 400,
        ]
    ).tolist()

    layers = [layernorm_all, linear_all, add_all, gelu_all, attention_all]

    layer_labels = ["layernorm", "linear", "add", "gelu", "attention"]

    for i, layer in enumerate(layers):

        layer_specific_stack = []

        current_dist = dists[d]

        mtx_per_block = np.zeros((len(layer), len(layer)))

        idx_1 = 0
        for k in layer:
            idx_2 = 0
            for l in layer:
                mtx_per_block[idx_1, idx_2] = current_dist[k, l]
                mtx_per_block[idx_2, idx_1] = current_dist[k, l]
                idx_2 += 1
            idx_1 += 1

        layer_specific_stack.append(mtx_per_block)

        plt.figure(figsize=(12, 8))
        plt.imshow(layer_specific_stack[0], "bone", vmin=0, vmax=np.max(layer_specific_stack[0]))
        plt.colorbar()
        plt.title(layer_labels[i] + " Layers " + dist_name.capitalize() + " Distance")
        ax = plt.gca()
        ax.set_axis_off()
        plt.savefig(
            os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/")
            + layer_labels[i]
            + "_specific_dist_"
            + dist_name
            + "_allvits_alllayers_inferencemode.svg",
            dpi=800,
        )

##### 2.1- Layer Retrieval Line Plot

In [ ]:
# Load them if necessary
# wass_dist = np.load(os.path.join(home_path, "Documents/BayesCompare/outputs/dist_covs_1000_vitb16_inferencemode_normalized_wasserstein.npy"))
# jsd_dist = np.load(os.path.join(home_path, "Documents/BayesCompare/outputs/dist_covs_1000_vitb16_inferencemode_normalized_JSD.npy"))

dists = [wass_dist, jsd_dist]
dist_names = ["wasserstein", "JSD"]

with open(
    os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/vit_inferencemode_layernames.json"),
    "r",
    encoding="utf-8",
) as f:
    model_layers = json.load(f)

model_names = ["1 2", "1 3", "1 4", "1 5", "2 3", "2 4", "2 5", "3 4", "3 5", "4 5"]

borders = [2, 10, 18, 26, 34, 42, 50, 58, 66, 74, 82, 90, 98]
borders_shifted = [i - 0.5 for i in borders]

for i, dist_name in enumerate(dist_names):

    m = 5  # number of models
    n = int(len(dists[i]) / m)  # number of layers per model

    blocks = dists[i].reshape((m, n, m, n)).transpose(0, 2, 1, 3).reshape(m * m, n, n)
    all_idx = np.triu(np.arange(m * m).reshape(m, m), k=1)
    idx = all_idx[np.where(all_idx != 0)]

    model_idx = 0

    cmap = plt.cm.get_cmap("viridis", 10)

    stack_diags = np.zeros((int(m * (m - 1) / 2), n))

    plt.figure(figsize=(17, 7), dpi=400)

    for i in idx:
        diags = np.diag(blocks[i])
        stack_diags[model_idx, :] = diags

        plt.plot(
            range(n),
            diags,
            ".-",
            linewidth=1,
            markersize=5,
            label="Model "
            + model_names[model_idx][0]
            + " vs Model "
            + model_names[model_idx][2],
            color=cmap(model_idx),
        )

        model_idx += 1

    ax = plt.gca()

    # remove the box around the plot
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Make y-axis start from 0
    ax.set_ylim(bottom=0)

    # Arrange y-axis ticks for each distance
    if dist_name == "JSD":

        ax.set_ylim(0, 1)
        ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1])

    elif dist_name == "wasserstein":

        ax.set_ylim(0, 5)
        ax.set_yticks([0, 1, 2, 3, 4, 5])
        ax.set_yticklabels(["0", "1", "2", "3", "4", "5"])

    # Place the legend to the left of the plot
    ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=8)

    # Set x-axis labels and ticks, and y-axis label
    plt.xlabel("Layers", fontsize=12)
    plt.xticks(ticks=range(n), labels=model_layers, rotation=90, fontsize=9)
    plt.ylabel(dist_name.capitalize() + " Distance", fontsize=12)

    # Arrange grid lines and layout
    plt.grid(axis="x", color="gray", alpha=0.3, linewidth=0.5)
    plt.tight_layout()

    ##--- You can either save the plot as it is
    plt.savefig(
        os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/compare_dist_")
        + dist_name
        + "_allvits_alllayers_inferencemode_retrieval.svg",
        dpi=400,
    )

    ##--- or you can add vertical lines corresponding to the block separations and save it with "wlines" in its name

    # for x0 in borders_shifted:
    #     ax.axvline(x=x0, color="red", linestyle="--", linewidth=1)

    # plt.tight_layout()
    # plt.savefig(
    #     os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/compare_dist_")
    #     + dist_name
    #     + "_allvits_alllayers_inferencemode_retrieval_wline.svg",
    #     dpi=400,
    # )

    # also, save the stacked diagonal values as a numpy array
    np.save(
        os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/stacked_diags")
        + dist_name
        + ".npy",
        stack_diags,
    )

##### 2.2- Layer Retrieval Line Plot without the 5th Model

In [ ]:
# Load them if necessary
# wass_dist = np.load(os.path.join(home_path, "Documents/BayesCompare/outputs/dist_covs_1000_vitb16_inferencemode_normalized_wasserstein.npy"))
# jsd_dist = np.load(os.path.join(home_path, "Documents/BayesCompare/outputs/dist_covs_1000_vitb16_inferencemode_normalized_JSD.npy"))

dists = [wass_dist, jsd_dist]
dist_names = ["wasserstein", "JSD"]

with open(
    os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/vit_inferencemode_layernames.json"),
    "r",
    encoding="utf-8",
) as f:
    model_layers = json.load(f)

model_names = ["1 2", "1 3", "1 4", "1 5", "2 3", "2 4", "2 5", "3 4", "3 5", "4 5"]

borders = [2, 10, 18, 26, 34, 42, 50, 58, 66, 74, 82, 90, 98]
borders_shifted = [i - 0.5 for i in borders]

fifth_model_idx = [4, 9, 14, 19]

for i, dist_name in enumerate(dist_names):

    m = 5  # number of models
    n = int(len(dists[i]) / m)  # number of layers per model

    blocks = dists[i].reshape((m, n, m, n)).transpose(0, 2, 1, 3).reshape(m * m, n, n)
    all_idx = np.triu(np.arange(m * m).reshape(m, m), k=1)
    idx = all_idx[np.where(all_idx != 0)]

    model_idx = 0

    colors_six = ["red", "orange", "gold", "green", "blue", "darkviolet"]

    stack_diags = np.zeros((int(m * (m - 1) / 2), n))

    plt.figure(figsize=(17, 7), dpi=400)

    for i in idx:
        diags = np.diag(blocks[i])
        stack_diags[model_idx, :] = diags

        if i in fifth_model_idx:
            model_idx += 1
            continue
        else:
            plt.plot(
                range(n),
                diags,
                ".-",
                linewidth=1,
                markersize=5,
                label="Model "
                + model_names[model_idx][0]
                + " vs Model "
                + model_names[model_idx][2],
                color=colors_six[color_idx],
            )
            color_idx += 1

        model_idx += 1

    ax = plt.gca()

    # remove the box around the plot
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Make y-axis start from 0
    ax.set_ylim(bottom=0)

    # Arrange y-axis ticks for each distance
    if dist_name == "JSD":

        ax.set_ylim(0, 1)
        ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1])

    elif dist_name == "wasserstein":

        ax.set_ylim(0, 5)
        ax.set_yticks([0, 1, 2, 3, 4, 5])
        ax.set_yticklabels(["0", "1", "2", "3", "4", "5"])

    # Place the legend to the left of the plot
    ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=8)

    # Set x-axis labels and ticks, and y-axis label
    plt.xlabel("Layers", fontsize=12)
    plt.xticks(ticks=range(n), labels=model_layers, rotation=90, fontsize=9)
    plt.ylabel(dist_name.capitalize() + " Distance", fontsize=12)

    # Arrange grid lines and layout
    plt.grid(axis="x", color="gray", alpha=0.3, linewidth=0.5)
    plt.tight_layout()

    ##--- You can either save the plot as it is
    plt.savefig(
        os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/compare_without_model5_dist_")
        + dist_name
        + "_allvits_alllayers_inferencemode_retrieval.svg",
        dpi=400,
    )

    ##--- or you can add vertical lines corresponding to the block separations and save it with "wlines" in its name

    # for x0 in borders_shifted:
    #     ax.axvline(x=x0, color="red", linestyle="--", linewidth=1)

    # plt.savefig(
    #     os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/compare_without_model5_dist_")
    #     + dist_name
    #     + "_allvits_alllayers_inferencemode_retrieval_wline.svg",
    #     dpi=400,
    # )

##### 2.3- Average Layer Retrieval Plot

You have to run `2.1- Layer Retrieval Line Plot` before running this part as `stacked_diags` arrays should be saved beforehand.

In [ ]:
# Load them if necessary
# wass_dist = np.load(os.path.join(home_path, "Documents/BayesCompare/outputs/dist_covs_1000_vitb16_inferencemode_normalized_wasserstein.npy"))
# jsd_dist = np.load(os.path.join(home_path, "Documents/BayesCompare/outputs/dist_covs_1000_vitb16_inferencemode_normalized_JSD.npy"))

dists = [wass_dist, jsd_dist]
dist_names = ["wasserstein", "JSD"]

with open(
    os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/vit_inferencemode_layernames.json"),
    "r",
    encoding="utf-8",
) as f:
    model_layers = json.load(f)

for i, dist_name in enumerate(dist_names):

    stacked_diags = np.load(
        os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/stacked_diags")
        + dist_name
        + ".npy"
    )

    # Get the mean and std from the stacked diagonals
    mean = stacked_diags.mean(axis=0)
    var = stacked_diags.var(axis=0, ddof=1)
    std = np.sqrt(var)

    plt.figure(figsize=(14, 6), dpi=400)
    plt.plot(range(n), mean, "-o", markersize=2, linewidth=1, color="black")
    plt.fill_between(range(n), mean - std, mean + std, alpha=0.15, color="gray")

    # Set x-axis labels and ticks, and y-axis label
    plt.xticks(ticks=range(n), labels=model_layers, rotation=90, fontsize=9)
    plt.xlabel("Layers", fontsize=12)
    plt.ylabel(dist_name.capitalize() + " Distance", fontsize=12)

    ax = plt.gca()

    # Remove the box around the plot
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Make y-axis start from 0
    ax.set_ylim(bottom=0)

    # Arrange y-axis ticks for each distance
    if dist_name == "JSD":

        ax.set_ylim(0, 1)
        ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1])

    elif dist_name == "wasserstein":

        ax.set_ylim(0, 5)
        ax.set_yticks([0, 1, 2, 3, 4, 5])
        ax.set_yticklabels(["0", "1", "2", "3", "4", "5"])

    # Arrange grid
    plt.grid(axis="x", color="gray", alpha=0.1, linewidth=0.5)
    plt.tight_layout()

    ##--- You can either save the plot as it is
    plt.savefig(
        os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/avg_dist_")
        + dist_name
        + "_allresnets_alllayers_inferencemode_retrieval.svg",
        dpi=400,
    )

    ##--- or you can add vertical lines corresponding to the block separations and save it with "wlines" in its name

    # for x0 in borders_shifted:
    #     ax.axvline(x=x0, color="red", linestyle="--", linewidth=1)

    # plt.savefig(
    #     os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/avg_dist_")
    #     + dist_name
    #     + "_allresnets_alllayers_inferencemode_retrieval_wline.svg",
    #     dpi=400,
    # )

##### 3.1- Kornblith Plots: Off-diagonal Blocks Averaged

In [ ]:
# Load them if necessary
# wass_dist = np.load(os.path.join(home_path, "Documents/BayesCompare/outputs/dist_covs_1000_vitb16_inferencemode_normalized_wasserstein.npy"))
# jsd_dist = np.load(os.path.join(home_path, "Documents/BayesCompare/outputs/dist_covs_1000_vitb16_inferencemode_normalized_JSD.npy"))

dists = [wass_dist, jsd_dist]
dist_names = ["wasserstein", "JSD"]

for i, dist_name in enumerate(dist_names):
    
    m = 5  # number of models
    n = int(len(dists[i]) / m)  # number of layers per model

    blocks = dists[i].reshape((m, n, m, n)).transpose(0, 2, 1, 3).reshape(m * m, n, n)
    all_idx = np.triu(np.arange(m * m).reshape(m, m), k=1)
    idx = all_idx[np.where(all_idx != 0)]
    
    stack_blocks = []

    for i in idx:
        stack_blocks.append(blocks[i])

    np_stack_blocks = np.array(stack_blocks)
    avg_stack_blocks = np.mean(np_stack_blocks, axis=0)

    plt.figure(figsize=(12, 8))
    plt.imshow(avg_stack_blocks, "bone", vmin=0, vmax=np.max(avg_stack_blocks))
    plt.colorbar()
    plt.title(dist_name.capitalize() + " Distance")
    ax = plt.gca()
    ax.set_axis_off()
    plt.savefig(
        os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/avged_blocks_dist_")
        + dist_name
        + "_allresnets_alllayers_inferencemode.svg",
        dpi=800,
    )

##### 3.2- Block Separated Kornblith Plots: Off-diagonal Blocks Averaged

In [ ]:
# Load them if necessary
# wass_dist = np.load(os.path.join(home_path, "Documents/BayesCompare/outputs/dist_covs_1000_vitb16_inferencemode_normalized_wasserstein.npy"))
# jsd_dist = np.load(os.path.join(home_path, "Documents/BayesCompare/outputs/dist_covs_1000_vitb16_inferencemode_normalized_JSD.npy"))

dists = [wass_dist, jsd_dist]
dist_names = ["wasserstein", "JSD"]

boundaries = [2, 10, 18, 26, 34, 42, 50, 58, 66, 74, 82, 90, 98]

N = 100 # one dimension of the matrix to be plotted
edges = [0] + boundaries + [N]  # [0, 2, 9, 17, ..., 97, 100]
group_sizes = np.diff(edges)  # e.g. [2, 7, 8, 8, ..., 3]
num_groups = len(group_sizes)

# (row_start, row_end), same for columnss
row_groups = [(edges[i], edges[i + 1]) for i in range(num_groups)]
col_groups = [(edges[j], edges[j + 1]) for j in range(num_groups)]

gap = 0.06  # spacing between blocks (tune this)
gs = gridspec.GridSpec(
    num_groups,
    num_groups,
    wspace=gap,
    hspace=gap,
    width_ratios=group_sizes,  # <-- scale width by #cols
    height_ratios=group_sizes,  # <-- scale height by #rows
)

for i, dist_name in enumerate(dist_names):

    m = 5  # number of models
    n = int(len(dists[i]) / m)  # number of layers per model

    blocks = dists[i].reshape((m, n, m, n)).transpose(0, 2, 1, 3).reshape(m * m, n, n)
    all_idx = np.triu(np.arange(m * m).reshape(m, m), k=1)
    idx = all_idx[np.where(all_idx != 0)]
    
    stack_blocks = []

    for i in idx:
        stack_blocks.append(blocks[i])

    np_stack_blocks = np.array(stack_blocks)
    avg_stack_blocks = np.mean(np_stack_blocks, axis=0)
    
    fig = plt.figure(figsize=(12, 12))

    for i, (r0, r1) in enumerate(row_groups):
        for j, (c0, c1) in enumerate(col_groups):
            ax = fig.add_subplot(gs[i, j])
            block = avg_stack_blocks[r0:r1, c0:c1]
            ax.imshow(block, cmap="bone", vmin=0, vmax=np.max(avg_stack_blocks))
            ax.set_xticks([])
            ax.set_yticks([])

            # remove the box around the matrix plot
            for spine in ax.spines.values():
                spine.set_visible(False)

    ax = plt.gca()
    ax.set_axis_off()
    plt.savefig(
        os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/avged_blocks_dist_")
        + dist_name
        + "_allresnets_alllayers_inferencemode_block_separated.svg",
        dpi=800,
    )

##### 3.3- Layer Specific Kornblith Plot

In [ ]:
# Load them if necessary
# wass_dist = np.load(
#     os.path.join(home_path, "Documents/BayesCompare/outputs/dist_covs_1000_vitb16_inferencemode_normalized_wasserstein_corrected.npy")
# )
# jsd_dist = np.load(
#     os.path.join(home_path, "Documents/BayesCompare/outputs/dist_covs_1000_vitb16_inferencemode_normalized_JSD_corrected.npy")
# )

dists = [wass_dist, jsd_dist]
dist_names = ["wasserstein", "JSD"]

# set this value specific for Wasserstein and JSD based on their overall colorbar scales from full Kornblith plots
colorbar_max_vals = [7.5, 1]

with open(
    os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/vit_inferencemode_layernames.json"),
    "r",
    encoding="utf-8",
) as f:
    model_layers = json.load(f)

for d, dist_name in enumerate(dist_names):

    m = 5  # number of models
    n = int(len(dists[d]) / m)  # number of layers per model

    layernorm_idx = []
    linear_idx = []
    add_idx = []
    gelu_idx = []
    attention_idx = []

    for i, layer_name in enumerate(model_layers):
        if "layernorm" in layer_name:
            layernorm_idx.append(i)
        elif "linear" in layer_name:
            linear_idx.append(i)
        elif "add" in layer_name:
            add_idx.append(i)
        elif "gelu" in layer_name:
            gelu_idx.append(i)
        elif "attention" in layer_name:
            attention_idx.append(i)

    layernorm_all = np.concatenate(
        [
            np.array(layernorm_idx),
            np.array(layernorm_idx) + 100,
            np.array(layernorm_idx) + 200,
            np.array(layernorm_idx) + 300,
            np.array(layernorm_idx) + 400,
        ]
    ).tolist()
    linear_all = np.concatenate(
        [
            np.array(linear_idx),
            np.array(linear_idx) + 100,
            np.array(linear_idx) + 200,
            np.array(linear_idx) + 300,
            np.array(linear_idx) + 400,
        ]
    ).tolist()
    add_all = np.concatenate(
        [
            np.array(add_idx),
            np.array(add_idx) + 100,
            np.array(add_idx) + 200,
            np.array(add_idx) + 300,
            np.array(add_idx) + 400,
        ]
    ).tolist()
    gelu_all = np.concatenate(
        [
            np.array(gelu_idx),
            np.array(gelu_idx) + 100,
            np.array(gelu_idx) + 200,
            np.array(gelu_idx) + 300,
            np.array(gelu_idx) + 400,
        ]
    ).tolist()
    attention_all = np.concatenate(
        [
            np.array(attention_idx),
            np.array(attention_idx) + 100,
            np.array(attention_idx) + 200,
            np.array(attention_idx) + 300,
            np.array(attention_idx) + 400,
        ]
    ).tolist()

    layers = [layernorm_all, linear_all, add_all, gelu_all, attention_all]

    layer_labels = ["layernorm", "linear", "add", "gelu", "attention"]

    for i, layer in enumerate(layers):

        layer_specific_stack = []

        current_dist = dists[d]
        
        layer_len = len(layer)

        mtx_per_block = np.zeros((layer_len, layer_len))

        idx_1 = 0
        for k in layer:
            idx_2 = 0
            for l in layer:
                mtx_per_block[idx_1, idx_2] = current_dist[k, l]
                mtx_per_block[idx_2, idx_1] = current_dist[k, l]
                idx_2 += 1
            idx_1 += 1

        layer_specific_stack.append(mtx_per_block)
        
        n_per_layer = int(layer_len/m)
        
        blocks = layer_specific_stack[0].reshape((m, n_per_layer, m, n_per_layer)).transpose(0, 2, 1, 3).reshape(m * m, n_per_layer, n_per_layer)
        all_idx = np.triu(np.arange(m * m).reshape(m, m), k=1)
        idx = all_idx[np.where(all_idx != 0)]
    
        stack_blocks = []
        
        for g in idx:
            stack_blocks.append(blocks[g])
            
        np_stack_blocks = np.array(stack_blocks)
        avg_stack_blocks = np.mean(np_stack_blocks, axis=0)

        plt.figure(figsize=(12, 8))
        # plt.imshow(
        #     avg_layer_specific, "bone", vmin=0, vmax=np.max(colorbar_max_vals[d])
        # )
        plt.imshow(avg_stack_blocks, "bone", vmin=0, vmax=np.max(avg_stack_blocks))
        plt.colorbar()
        plt.title(layer_labels[i] + " Layers " + dist_name.capitalize() + " Distance")
        ax = plt.gca()
        ax.set_axis_off()
        plt.savefig(
            os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/layer_specific_analysis/")
            + layer_labels[i]
            + "_specific_avged_dist_"
            + dist_name
            + "_allvits_alllayers_inferencemode.svg",
            dpi=800,
        )

##### 4.1- Layer Specific MDS Plots

In [ ]:
from sklearn.manifold import MDS

# Load them if necessary
# wass_dist = np.load(
#     os.path.join(home_path, "Documents/BayesCompare/outputs/dist_covs_1000_vitb16_inferencemode_normalized_wasserstein_corrected.npy")
# )
# jsd_dist = np.load(
#     os.path.join(home_path, "Documents/BayesCompare/outputs/dist_covs_1000_vitb16_inferencemode_normalized_JSD_corrected.npy")
# )

dists = [wass_dist, jsd_dist]
dist_names = ["wasserstein", "JSD"]

# set this value specific for Wasserstein and JSD based on their overall colorbar scales from full Kornblith plots
colorbar_max_vals = [7.5, 1]

with open(
    os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/vit_inferencemode_layernames.json"),
    "r",
    encoding="utf-8",
) as f:
    model_layers = json.load(f)

for d, dist_name in enumerate(dist_names):

    m = 5  # number of models
    n = int(len(dists[d]) / m)  # number of layers per model

    layernorm_idx = []
    linear_idx = []
    add_idx = []
    gelu_idx = []
    attention_idx = []

    for i, layer_name in enumerate(model_layers):
        if "layernorm" in layer_name:
            layernorm_idx.append(i)
        elif "linear" in layer_name:
            linear_idx.append(i)
        elif "add" in layer_name:
            add_idx.append(i)
        elif "gelu" in layer_name:
            gelu_idx.append(i)
        elif "attention" in layer_name:
            attention_idx.append(i)

    layernorm_all = np.concatenate(
        [
            np.array(layernorm_idx),
            np.array(layernorm_idx) + 100,
            np.array(layernorm_idx) + 200,
            np.array(layernorm_idx) + 300,
            np.array(layernorm_idx) + 400,
        ]
    ).tolist()
    linear_all = np.concatenate(
        [
            np.array(linear_idx),
            np.array(linear_idx) + 100,
            np.array(linear_idx) + 200,
            np.array(linear_idx) + 300,
            np.array(linear_idx) + 400,
        ]
    ).tolist()
    add_all = np.concatenate(
        [
            np.array(add_idx),
            np.array(add_idx) + 100,
            np.array(add_idx) + 200,
            np.array(add_idx) + 300,
            np.array(add_idx) + 400,
        ]
    ).tolist()
    gelu_all = np.concatenate(
        [
            np.array(gelu_idx),
            np.array(gelu_idx) + 100,
            np.array(gelu_idx) + 200,
            np.array(gelu_idx) + 300,
            np.array(gelu_idx) + 400,
        ]
    ).tolist()
    attention_all = np.concatenate(
        [
            np.array(attention_idx),
            np.array(attention_idx) + 100,
            np.array(attention_idx) + 200,
            np.array(attention_idx) + 300,
            np.array(attention_idx) + 400,
        ]
    ).tolist()

    layers = [layernorm_all, linear_all, add_all, gelu_all, attention_all]

    layer_labels = ["layernorm", "linear", "add", "gelu", "attention"]

    for i, layer in enumerate(layers):

        layer_specific_stack = []

        current_dist = dists[d]

        mtx_per_block = np.zeros((len(layer), len(layer)))

        idx_1 = 0
        for k in layer:
            idx_2 = 0
            for l in layer:
                mtx_per_block[idx_1, idx_2] = current_dist[k, l]
                mtx_per_block[idx_2, idx_1] = current_dist[k, l]
                idx_2 += 1
            idx_1 += 1

        layer_specific_stack.append(mtx_per_block)
        
        
        n_per_layer = int(len(layer)/m)  # number of layers per model per specific layer
        
        mds = MDS(dissimilarity="precomputed")
        mds.fit(layer_specific_stack[0])

        x = mds.embedding_
        x0 = x[0:n_per_layer]
        x1 = x[n_per_layer:2*n_per_layer]
        x2 = x[2*n_per_layer:3*n_per_layer]
        x3 = x[3*n_per_layer:4*n_per_layer]
        x4 = x[4*n_per_layer:5*n_per_layer]

        plt.figure()
        plt.plot(x0[:, 0], x0[:, 1], '.-', linewidth=2, markersize=10, color="#fa5750", label="Model 1")  
        plt.plot(x1[:, 0], x1[:, 1], '.-', linewidth=2, markersize=10, color="#dbb32d", label="Model 2")  
        plt.plot(x2[:, 0], x2[:, 1], '.-', linewidth=2, markersize=10, color="#4695f7", label="Model 3")  
        plt.plot(x3[:, 0], x3[:, 1], '.-', linewidth=2, markersize=10, color="#33db2d", label="Model 4") 
        plt.plot(x4[:, 0], x4[:, 1], '.-', linewidth=2, markersize=10, color="#d446f7", label="Model 5") 
        
        # Add starting dots
        plt.plot(x0[0,0], x0[0,1], 'o', markersize=8, color="#862724")
        plt.plot(x1[0,0], x1[0,1], 'o', markersize=8, color="#6b5612")
        plt.plot(x2[0,0], x2[0,1], 'o', markersize=8, color="#10345f")
        plt.plot(x3[0,0], x3[0,1], 'o', markersize=8, color="#126110")
        plt.plot(x4[0,0], x4[0,1], 'o', markersize=8, color="#4e125e")
        
        plt.text(0.01, 0.99, "Layer: "+layer_labels[i]+" - Distance: "+dist_name, transform=plt.gca().transAxes,
         ha='left', va='top', fontsize=8)
        
        plt.legend(loc="lower right", fontsize=8)
        
        plt.axis("equal")
        ax = plt.gca()
        ax.set_axis_off()
        
        plt.savefig(
            os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/layer_specific_analysis/MDS_"+layer_labels[i]+"_specific_avged_blocks_dist_")
            + dist_name
            + "_allresnets_alllayers_inferencemode.svg",
            dpi=800,
        )

##### 4.2- Position Based Layer Specific MDS Plots

In [ ]:
from sklearn.manifold import MDS

# Load them if necessary
# wass_dist = np.load(
#     os.path.join(home_path, "Documents/BayesCompare/outputs/dist_covs_1000_vitb16_inferencemode_normalized_wasserstein_corrected.npy")
# )
# jsd_dist = np.load(
#     os.path.join(home_path, "Documents/BayesCompare/outputs/dist_covs_1000_vitb16_inferencemode_normalized_JSD_corrected.npy")
# )

dists = [wass_dist, jsd_dist]
dist_names = ["wasserstein", "JSD"]

# set this value specific for Wasserstein and JSD based on their overall colorbar scales from full Kornblith plots
colorbar_max_vals = [7.5, 1]

with open(
    os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/vit_inferencemode_layernames.json"),
    "r",
    encoding="utf-8",
) as f:
    model_layers = json.load(f)

for d, dist_name in enumerate(dist_names):

    m = 5  # number of models
    n = int(len(dists[d]) / m)  # number of layers per model

    layernorm_idx = []
    linear_idx = []
    add_idx = []

    for i, layer_name in enumerate(model_layers):
        if "layernorm" in layer_name:
            layernorm_idx.append(i)
        elif "linear" in layer_name:
            linear_idx.append(i)
        elif "add" in layer_name:
            add_idx.append(i)

        layernorm_1 = layernorm_idx[0::2]
        layernorm_2 = layernorm_idx[1::2]

        add_1 = add_idx[0::2]
        add_2 = add_idx[1::2]

        linear_1 = linear_idx[0::2]
        linear_2 = linear_idx[1::2]

    layernorm_all_1 = np.concatenate(
        [
            np.array(layernorm_1),
            np.array(layernorm_1) + 100,
            np.array(layernorm_1) + 200,
            np.array(layernorm_1) + 300,
            np.array(layernorm_1) + 400,
        ]
    ).tolist()

    layernorm_all_2 = np.concatenate(
        [
            np.array(layernorm_2),
            np.array(layernorm_2) + 100,
            np.array(layernorm_2) + 200,
            np.array(layernorm_2) + 300,
            np.array(layernorm_2) + 400,
        ]
    ).tolist()

    add_all_1 = np.concatenate(
        [
            np.array(add_1),
            np.array(add_1) + 100,
            np.array(add_1) + 200,
            np.array(add_1) + 300,
            np.array(add_1) + 400,
        ]
    ).tolist()

    add_all_2 = np.concatenate(
        [
            np.array(add_2),
            np.array(add_2) + 100,
            np.array(add_2) + 200,
            np.array(add_2) + 300,
            np.array(add_2) + 400,
        ]
    ).tolist()

    linear_all_1 = np.concatenate(
        [
            np.array(linear_1),
            np.array(linear_1) + 100,
            np.array(linear_1) + 200,
            np.array(linear_1) + 300,
            np.array(linear_1) + 400,
        ]
    ).tolist()

    linear_all_2 = np.concatenate(
        [
            np.array(linear_2),
            np.array(linear_2) + 100,
            np.array(linear_2) + 200,
            np.array(linear_2) + 300,
            np.array(linear_2) + 400,
        ]
    ).tolist()

    layers = [
        layernorm_all_1,
        layernorm_all_2,
        linear_all_1,
        linear_all_2,
        add_all_1,
        add_all_2,
    ]

    layer_labels = [
        "layernorm_1",
        "layernorm_2",
        "linear_1",
        "linear_2",
        "add_1",
        "add_2",
    ]

    for i, layer in enumerate(layers):

        layer_specific_stack = []

        current_dist = dists[d]

        mtx_per_block = np.zeros((len(layer), len(layer)))

        idx_1 = 0
        for k in layer:
            idx_2 = 0
            for l in layer:
                mtx_per_block[idx_1, idx_2] = current_dist[k, l]
                mtx_per_block[idx_2, idx_1] = current_dist[k, l]
                idx_2 += 1
            idx_1 += 1

        layer_specific_stack.append(mtx_per_block)

        n_per_layer = int(
            len(layer) / m
        )  # number of layers per model per specific layer

        mds = MDS(dissimilarity="precomputed")
        mds.fit(layer_specific_stack[0])

        x = mds.embedding_
        x0 = x[0:n_per_layer]
        x1 = x[n_per_layer : 2 * n_per_layer]
        x2 = x[2 * n_per_layer : 3 * n_per_layer]
        x3 = x[3 * n_per_layer : 4 * n_per_layer]
        x4 = x[4 * n_per_layer : 5 * n_per_layer]

        plt.figure()
        plt.plot(
            x0[:, 0],
            x0[:, 1],
            ".-",
            linewidth=2,
            markersize=10,
            color="#fa5750",
            label="Model 1",
        )
        plt.plot(
            x1[:, 0],
            x1[:, 1],
            ".-",
            linewidth=2,
            markersize=10,
            color="#dbb32d",
            label="Model 2",
        )
        plt.plot(
            x2[:, 0],
            x2[:, 1],
            ".-",
            linewidth=2,
            markersize=10,
            color="#4695f7",
            label="Model 3",
        )
        plt.plot(
            x3[:, 0],
            x3[:, 1],
            ".-",
            linewidth=2,
            markersize=10,
            color="#33db2d",
            label="Model 4",
        )
        plt.plot(
            x4[:, 0],
            x4[:, 1],
            ".-",
            linewidth=2,
            markersize=10,
            color="#d446f7",
            label="Model 5",
        )

        # Add starting dots
        plt.plot(x0[0, 0], x0[0, 1], "o", markersize=8, color="#862724")
        plt.plot(x1[0, 0], x1[0, 1], "o", markersize=8, color="#6b5612")
        plt.plot(x2[0, 0], x2[0, 1], "o", markersize=8, color="#10345f")
        plt.plot(x3[0, 0], x3[0, 1], "o", markersize=8, color="#126110")
        plt.plot(x4[0, 0], x4[0, 1], "o", markersize=8, color="#4e125e")

        plt.text(
            0.01,
            0.99,
            "Layer: " + layer_labels[i]+" - Distance: "+dist_name,
            transform=plt.gca().transAxes,
            ha="left",
            va="top",
            fontsize=8,
        )

        plt.legend(loc="lower right", fontsize=8)

        plt.axis("equal")
        ax = plt.gca()
        ax.set_axis_off()

        plt.savefig(
            os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/layer_specific_analysis/MDS_position_based_")
            + layer_labels[i]
            + "_specific_avged_blocks_dist_"
            + dist_name
            + "_allresnets_alllayers_inferencemode.svg",
            dpi=800,
        )


##### 5- Layer Types' Mean and Variance Comparison - Violin Plot

In [ ]:
jsd_dist_stacked = np.load(
    "/home/sezan/Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/stacked_diagsJSD.npy"
)
wass_dist_stacked = np.load(
    "/home/sezan/Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/stacked_diagswasserstein.npy"
)

with open(
    os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/vit_inferencemode_layernames.json"),
    "r",
    encoding="utf-8",
) as f:
    model_layers = json.load(f)

layernorm_1_idx = []
layernorm_2_idx = []
linear_1_idx = []
linear_2_idx = []
add_1_idx = []
add_2_idx = []
add_idx = []
gelu_idx = []
attention_idx = []
conv_idx = []

for i, layer_name in enumerate(model_layers):
    if "layernorm" in layer_name and int(layer_name[10:11]) % 2 == 1:
        layernorm_1_idx.append(i)
    elif "layernorm" in layer_name and int(layer_name[10:11]) % 2 == 0:
        layernorm_2_idx.append(i)
    elif "linear" in layer_name and int(layer_name[7:8]) % 2 == 1:
        linear_1_idx.append(i)
    elif "linear" in layer_name and int(layer_name[7:8]) % 2 == 0:
        linear_2_idx.append(i)
    elif "add" in layer_name and layer_name[-1:] == "1":
        add_1_idx.append(i)
    elif "add" in layer_name and layer_name[-1:] == "2":
        add_2_idx.append(i)
    elif "add" in layer_name:
        add_idx.append(i)
    elif "gelu" in layer_name:
        gelu_idx.append(i)
    elif "attention" in layer_name:
        attention_idx.append(i)
    elif "conv" in layer_name:
        conv_idx.append(i)

def to_rows(arr, measure, layer):
    return [(measure, layer, float(v)) for v in np.asarray(arr).ravel()]


def build_df_from_stacks(
    stacks_by_measure: dict[str, np.ndarray],
    layer_to_idx: dict[str, int],
    measures: list[str],
    layer_order: list[str],   # the canonical layer order you want in x
):
    # --- build rows ---
    rows = []
    for m in measures:
        S = stacks_by_measure[m]
        for layer in layer_order:
            rows += to_rows(S[:, layer_to_idx[layer]], m, layer)

    df = pd.DataFrame(rows, columns=["measure", "layer", "value"])
    df["x"] = df["measure"] + "_" + df["layer"]

    x_order = [f"{m}_{layer}" for m in measures for layer in layer_order]
    df["x"] = pd.Categorical(df["x"], categories=x_order, ordered=True)
    return df, x_order

measures = ["Wasserstein", "JSD"] 

stacks_by_measure = {
    "Wasserstein": wass_dist_stacked,
    "JSD": jsd_dist_stacked,
}


layer_to_idx = {
    "layernorm_1": layernorm_1_idx,
    "layernorm_2": layernorm_2_idx,
    "linear_1": linear_1_idx,
    "linear_2":linear_2_idx,
    "add_1": add_1_idx,
    "add_2": add_2_idx,
    "add": add_idx,
    "gelu": gelu_idx,
    "attention": attention_idx,
    "conv": conv_idx
}

layer_order = list(layer_to_idx.keys()) 

df, order = build_df_from_stacks(
    stacks_by_measure=stacks_by_measure,
    layer_to_idx=layer_to_idx,
    measures=measures,
    layer_order=layer_order,
)

cat_order = [["add", "conv", "layernorm_1", "layernorm_2", "gelu", "linear_1", "add_1", "add_2", "linear_2", "attention"],
        ["add", "conv", "layernorm_1", "layernorm_2", "gelu", "linear_1", "linear_2", "add_1", "add_2", "attention"]]

fig, axes = plt.subplots(1, 2, figsize=(28, 10))

for ax, meas, palette, id in zip(
    axes,
    measures,
    ["red", "blue"], [0, 1]
):
    d = df[df["measure"] == meas].copy()
 
    d["layer"] = pd.Categorical(d["layer"], categories=cat_order[id], ordered=True)
 
    # --- swarm points ---
    sns.swarmplot(data=d, x="layer", y="value", order=cat_order[id],
                  color=palette, size=4, ax=ax)
    sns.violinplot(data=d, x="layer", y="value", order=cat_order[id],
    color=palette, inner=None, linewidth=0, cut=0, ax=ax, alpha=0.25)
    
    means = (
        d.groupby("layer")["value"].mean()
         .reindex(cat_order[id])          
    )
 
    ax.scatter(
        means.index, means.values,
        s=120, marker="o",       
        facecolor="gray", edgecolor="black", linewidth=1.6,
        zorder=10, label="Mean"
    )
 
    # clean axis labels
    ax.set_xlabel("")                   
    ax.set_ylabel("Distance")
    ax.set_xticklabels(cat_order[id])
    ax.yaxis.grid(True, which='major', color='lightgray', linestyle='-', linewidth=0.7)
    ax.set_axisbelow(True)
 
    # add group label UNDER the ticks
    fig.subplots_adjust(bottom=0.22)
    ax.text(0.5, -0.03, meas, transform=ax.transAxes,
            ha="center", va="top", fontsize=12)
 
plt.tight_layout()
plt.savefig(os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/layers_mean_var_comparison_allvits_alllayers_inferencemode.svg"),
            dpi=800)

##### 6.1- Layer Specific Retrieval Plots - Altogether

In [ ]:
jsd_dist_stacked = np.load(
    os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/stacked_diagsJSD.npy")
)
wass_dist_stacked = np.load(
   os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/stacked_diagswasserstein.npy")
)

with open(
    os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/vit_inferencemode_layernames.json"),
    "r",
    encoding="utf-8",
) as f:
    model_layers = json.load(f)

pattern = re.compile(r"^([a-zA-Z]+)_(\d{1,3})_(\d{1,3})$")

def split_string(s):
    m = pattern.match(s)
    if not m:
        return [None, None, None]
    return m.group(1), m.group(2), m.group(3)

layernorm_1_idx = []
layernorm_2_idx = []
linear_1_idx = []
linear_2_idx = []
add_1_idx = []
add_2_idx = []
gelu_idx = []
attention_idx = []

for i, layer_name in enumerate(model_layers):

    part1, part2, part3 = split_string(layer_name)

    if (
        "layernorm" in layer_name
        and int(part2) % 2 == 1
        and layer_name != "layernorm_25_128"
    ):
        layernorm_1_idx.append(i)
    elif (
        "layernorm" in layer_name
        and int(part2) % 2 == 0
        and layer_name != "layernorm_25_128"
    ):
        layernorm_2_idx.append(i)
    elif (
        "linear" in layer_name and int(part2) % 2 == 1 and layer_name != "linear_25_130"
    ):
        linear_1_idx.append(i)
    elif (
        "linear" in layer_name and int(part2) % 2 == 0 and layer_name != "linear_25_130"
    ):
        linear_2_idx.append(i)
    elif "add" in layer_name and layer_name[-1:] == "1":
        add_1_idx.append(i)
    elif "add" in layer_name and layer_name[-1:] == "2":
        add_2_idx.append(i)
    elif "gelu" in layer_name:
        gelu_idx.append(i)
    elif "attention" in layer_name:
        attention_idx.append(i)

layers = [layernorm_1_idx, attention_idx, add_1_idx, layernorm_2_idx, linear_1_idx, gelu_idx, linear_2_idx, add_2_idx]
layer_labels = ["layernorm_1", "attention", "add_1", "layernorm_2", "linear_1", "gelu", "linear_2", "add_2"]
colors = [
    '#e41a1c',  # red
    '#377eb8',  # blue
    '#4daf4a',  # green
    '#984ea3',  # purple
    '#ff7f00',  # orange
    '#a65628',  # brown
    '#f781bf',  # pink
    '#999999',  # gray
]

colors_std = [
    '#f6b6b6',  # light red
    '#c6d8f0',  # light blue
    '#cfe8cf',  # light green
    '#e0cde9',  # light purple
    '#ffe0b3',  # light orange
    '#e2cbb8',  # light brown
    '#f7d3e5',  # light pink
    '#dddddd',  # light gray
]

measures = ["Wasserstein", "JSD"] 
stacks = [wass_dist_stacked, jsd_dist_stacked]

for d, dist_name in enumerate(measures):
    
    fig, ax = plt.subplots(figsize=(14, 6), dpi=400)
    
    for i, layer in enumerate(layers):
        
        stack = stacks[d]
        layer_stack = stack[:, layer]
        mean = layer_stack.mean(axis=0)
        var = layer_stack.var(axis=0, ddof=1)
        std = np.sqrt(var)
        
        n = len(mean)
        
        plt.plot(range(n), mean, '-o', markersize=2, linewidth=1, color=colors[i], label=layer_labels[i])
        plt.fill_between(range(n), mean-std, mean+std, alpha=0.15, color=colors_std[i])

    xlabels = [str(i) for i in range(1, 13)]
        
    plt.xticks(ticks=range(n), labels=xlabels, fontsize=6)
    
    plt.xlabel("Block Number")
    plt.ylabel(dist_name.capitalize()+" Distance")
    plt.legend()
    plt.tight_layout()
    plt.grid(axis='x', color='gray', alpha=0.1, linewidth=0.5)
    plt.savefig(os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/layer_specific_analysis/per_layer_type_retrieval_"+ dist_name
            +"allvits_alllayers_inferencemode.svg"),
            dpi=800)


##### 6.2- Layer Specific Retrieval Plots - Layers Separated

In [ ]:
jsd_dist_stacked = np.load(
    os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/stacked_diagsJSD.npy")
)
wass_dist_stacked = np.load(
   os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/stacked_diagswasserstein.npy")
)

with open(
    os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/vit_inferencemode_layernames.json"),
    "r",
    encoding="utf-8",
) as f:
    model_layers = json.load(f)

layernorm_1_idx = []
layernorm_2_idx = []
linear_1_idx = []
linear_2_idx = []
add_1_idx = []
add_2_idx = []
gelu_idx = []
attention_idx = []

for i, layer_name in enumerate(model_layers):

    part1, part2, part3 = split_string(layer_name)

    if (
        "layernorm" in layer_name
        and int(part2) % 2 == 1
        and layer_name != "layernorm_25_128"
    ):
        layernorm_1_idx.append(i)
    elif (
        "layernorm" in layer_name
        and int(part2) % 2 == 0
        and layer_name != "layernorm_25_128"
    ):
        layernorm_2_idx.append(i)
    elif (
        "linear" in layer_name and int(part2) % 2 == 1 and layer_name != "linear_25_130"
    ):
        linear_1_idx.append(i)
    elif (
        "linear" in layer_name and int(part2) % 2 == 0 and layer_name != "linear_25_130"
    ):
        linear_2_idx.append(i)
    elif "add" in layer_name and layer_name[-1:] == "1":
        add_1_idx.append(i)
    elif "add" in layer_name and layer_name[-1:] == "2":
        add_2_idx.append(i)
    elif "gelu" in layer_name:
        gelu_idx.append(i)
    elif "attention" in layer_name:
        attention_idx.append(i)

layers = [layernorm_1_idx, attention_idx, add_1_idx, layernorm_2_idx, linear_1_idx, gelu_idx, linear_2_idx, add_2_idx]
layer_labels = ["layernorm_1", "attention", "add_1", "layernorm_2", "linear_1", "gelu", "linear_2", "add_2"]
colors = [
    '#e41a1c',  # red
    '#377eb8',  # blue
    '#4daf4a',  # green
    '#984ea3',  # purple
    '#ff7f00',  # orange
    '#a65628',  # brown
    '#f781bf',  # pink
    '#999999',  # gray
]

colors_std = [
    '#f6b6b6',  # light red
    '#c6d8f0',  # light blue
    '#cfe8cf',  # light green
    '#e0cde9',  # light purple
    '#ffe0b3',  # light orange
    '#e2cbb8',  # light brown
    '#f7d3e5',  # light pink
    '#dddddd',  # light gray
]

measures = ["Wasserstein", "JSD"] 
stacks = [wass_dist_stacked, jsd_dist_stacked]

for d, dist_name in enumerate(measures):
    
    for i, layer in enumerate(layers):
        
        fig, ax = plt.subplots(figsize=(14, 6), dpi=400)
        
        stack = stacks[d]
        layer_stack = stack[:, layer]
        mean = layer_stack.mean(axis=0)
        var = layer_stack.var(axis=0, ddof=1)
        std = np.sqrt(var)
        
        n = len(mean)
        
        plt.plot(range(n), mean, '-o', markersize=2, linewidth=1, color=colors[i], label=layer_labels[i])
        plt.fill_between(range(n), mean-std, mean+std, alpha=0.15, color=colors_std[i])

        xlabels = [str(i) for i in range(1, 13)]
            
        plt.xticks(ticks=range(n), labels=xlabels, fontsize=6)
        plt.title(layer_labels[i])
        plt.xlabel("Block Number")
        plt.ylabel(dist_name.capitalize()+" Distance")
        plt.tight_layout()
        plt.grid(axis='x', color='gray', alpha=0.1, linewidth=0.5)
        plt.savefig(os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/layer_specific_analysis/"+layer_labels[i]+"_retrieval_"+ dist_name
                +"allvits_alllayers_inferencemode.svg"),
                dpi=800)


##### 7.1- Wasserstein vs JSD Plot - Color Coding wrt Layer Depth

In [ ]:
from matplotlib.colors import Normalize

def layer_depth_plot(inp1, inp2, order=["JSD", "Wasserstein"]):

    x = inp1.flatten()
    y = inp2.flatten()

    # keep only positive values (required for log scale)
    mask = (x > 0) & (y > 0)
    x = x[mask]
    y = y[mask]
    
    G, S = inp1.shape  # G=10, S=176
    layer_idx = np.tile(np.arange(S), G)

    norm_layer = Normalize(vmin=np.min(layer_idx), vmax=np.max(layer_idx))

    fig, ax = plt.subplots(figsize=(13, 10))
    sc = ax.scatter(
        x,
        y,
        c=layer_idx,
        s=20,
        alpha=0.9,
        cmap="plasma",
        norm=norm_layer,
    )

    ax.set_xlabel(order[0])
    ax.set_ylabel(order[1])
    ax.set_title("Layer Depth")
    plt.colorbar(sc, ax=ax)

    plt.savefig(
        os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/jsd_vs_wass_layer_depth.svg"),
        dpi=800,
    )
    
jsd_dist_stacked = np.load(
    os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/stacked_diagsJSD.npy")
)
wass_dist_stacked = np.load(
   os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/stacked_diagswasserstein.npy")
)

layer_depth_plot(jsd_dist_stacked, wass_dist_stacked)


##### 7.2- Wasserstein vs JSD Plot - Color Coding wrt Model Pairs

In [ ]:
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D

def model_pair_plot(inp1, inp2, order=["JSD", "Wasserstein"]):

    x = inp1.flatten()
    y = inp2.flatten()

    # keep only positive values (required for log scale)
    mask = (x > 0) & (y > 0)
    x = x[mask]
    y = y[mask]

    model_pair_names = ["1 2", "1 3", "1 4", "1 5", "2 3", "2 4", "2 5", "3 4", "3 5", "4 5"]

    G, S = inp1.shape  # G=10, S=176
    group_idx = np.repeat(np.arange(G), S)

    cmap = plt.get_cmap("tab10")  # good discrete colormap
    colors = ListedColormap([cmap(i % len(model_pair_names)) for i in range(G)])

    fig, ax = plt.subplots(figsize=(10, 10))
    sc = ax.scatter(
        x,
        y,
        c=group_idx,
        s=20,
        alpha=0.9,
        cmap=colors,
    )

    ax.set_xlabel(order[0])
    ax.set_ylabel(order[1])

    labels = [
        "Model " + model_pair_names[g][0] + " vs Model " + model_pair_names[g][2]
        for g in range(len(model_pair_names))
    ]

    label_to_color = {labels[i]: colors.colors[i] for i in range(len(labels))}

    legend_elements = [
        Line2D(
            [0],
            [0],
            marker="o",
            color="none",
            markerfacecolor=label_to_color[k],
            markersize=8,
            label=f"{k}",
        )
        for k in label_to_color
    ]

    ax.legend(handles=legend_elements, title="Model Pairs")

    plt.savefig(
        os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/jsd_vs_wass_model_pair.svg"),
        dpi=800,
    )

jsd_dist_stacked = np.load(
    os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/stacked_diagsJSD.npy")
)
wass_dist_stacked = np.load(
   os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/stacked_diagswasserstein.npy")
)

model_pair_plot(jsd_dist_stacked, wass_dist_stacked)

##### 7.3- Wasserstein vs JSD Plot - Color Coding wrt Layer Type

In [ ]:
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D

def layer_type_plot(inp1, inp2, order=["JSD", "Wasserstein"]):

    x = inp1.flatten()
    y = inp2.flatten()

    # keep only positive values (required for log scale)
    mask = (x > 0) & (y > 0)
    x = x[mask]
    y = y[mask]

    with open(
        os.path.join(
            home_path,
            "Documents/BayesCompare/figures/vit_b_16_results/vit_inferencemode_layernames.json",
        ),
        "r",
        encoding="utf-8",
    ) as f:
        model_layers = json.load(f)

    layer_groups = []

    for layer_name in model_layers:
        
        part1, part2, part3 = split_string(layer_name)

        if (
            "layernorm" in layer_name
            and int(part2) % 2 == 1
        ):
            layer_groups.append(0)
        elif (
            "layernorm" in layer_name
            and int(part2) % 2 == 0
        ):
            layer_groups.append(1)
        elif (
        "linear" in layer_name and int(part2) % 2 == 1
        ):
            layer_groups.append(2)
        elif (
        "linear" in layer_name and int(part2) % 2 == 0
        ):
            layer_groups.append(3)
        elif "add" in layer_name and layer_name[-1:] == "1":
            layer_groups.append(4)
        elif "add" in layer_name and layer_name[-1:] == "2":
            layer_groups.append(5)
        elif "add" in layer_name:
            layer_groups.append(6)
        elif "gelu" in layer_name:
            layer_groups.append(7)
        elif "attention" in layer_name:
            layer_groups.append(8)
        elif "conv" in layer_name:
            layer_groups.append(9)

    G, S = inp1.shape  # G=10, S=176
    layer_groups_idx = np.tile(layer_groups, G)

    cmap = plt.get_cmap("tab10")  # good discrete colormap
    colors = ListedColormap([cmap(i % 10) for i in range(10)])

    fig, ax = plt.subplots(figsize=(10, 10))
    sc = ax.scatter(
        x,
        y,
        c=layer_groups_idx,
        s=20,
        alpha=0.9,
        cmap=colors,
    )

    ax.set_xlabel(order[0])
    ax.set_ylabel(order[1])

    labels = [
        "layernorm_1",
        "layernorm_2",
        "linear_1",
        "linear_2",
        "add_1",
        "add_2",
        "add",
        "gelu",
        "attention",
        "conv",
    ]

    label_to_color = {labels[i]: colors.colors[i] for i in range(len(labels))}

    legend_elements = [
        Line2D(
            [0],
            [0],
            marker="o",
            color="none",
            markerfacecolor=label_to_color[k],
            markersize=8,
            label=f"{k}",
        )
        for k in label_to_color
    ]

    ax.legend(handles=legend_elements)

    plt.savefig(
        os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/jsd_vs_wass_layer_type.svg"),
        dpi=800,
    )
    
jsd_dist_stacked = np.load(
    os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/stacked_diagsJSD.npy")
)
wass_dist_stacked = np.load(
   os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/stacked_diagswasserstein.npy")
)

layer_type_plot(jsd_dist_stacked, wass_dist_stacked)

##### 7.4- Wasserstein vs JSD Plot - Color Coding wrt Training Variation Type

In [ ]:
colors = ['b', 'g', 'r']

var_names = ["seed", "seed+aug", "pre-trained vs home-trained"]

jsd_dist_stacked = np.load(
    os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/stacked_diagsJSD.npy")
)
wass_dist_stacked = np.load(
   os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/stacked_diagswasserstein.npy")
)

model_pair_names = ["1 2", "1 3", "1 4", "1 5", "2 3", "2 4", "2 5", "3 4", "3 5", "4 5"]

x = jsd_dist_stacked.reshape(-1)
y = wass_dist_stacked.reshape(-1)

G, S = jsd_dist_stacked.shape  # G=10, S=176
group_idx = np.repeat(np.arange(G), S)

plt.figure(figsize=(10,10))
for g in range(G):
    if g == 0:
        label = var_names[0]
        col = colors[0]
    elif g in [1, 2, 4, 5, 7]:
        if g ==1:
            label = var_names[1]
        else:
            label= None
        col = colors[1]
    elif g in [3, 6, 8, 9]:
        if g==3:
            label = var_names[2]
        else:
            label= None
        col = colors[2]
        
    mask = group_idx == g
    plt.scatter(x[mask], y[mask], color=col, s=20, alpha=0.9, label=label)
 
plt.xlabel('JSD')
plt.ylabel('Wasserstein')
plt.legend(title='Variation Types', frameon=True, loc='best')
plt.tight_layout()
plt.savefig(
        os.path.join(home_path, "Documents/BayesCompare/figures/vit_b_16_results/all_layers_inferencemode/jsd_vs_wass_variation_type.svg"),
        dpi=800,
    )